# 🚦 Flipkart Gridlock Hackathon — Traffic Demand Prediction

**Problem**: Predict traffic demand (a normalized float in [0,1]) for each location (geohash) at each timestamp.

**Evaluation metric**: `score = max(0, 100 * R²_score(actual, predicted))`

**Approach**: LightGBM + XGBoost ensemble with rich historical lag features.

---

## Strategy Overview

After deep analysis of the dataset:
1. **Geo-temporal historical demand** is the single strongest signal (same location, same time on previous day)
2. **RoadType** is a hugely impactful categorical (Highway: 0.61, Street: 0.27, Residential: 0.057 mean demand)
3. We use a **gradient boosting ensemble** with rich aggregated features
4. Missing values are handled via imputation and fallback chains

In [ ]:
# ============================================================
# CELL 1: Install and Import Libraries
# ============================================================

# Install required packages if not available
import subprocess, sys
def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

try:
    import lightgbm
except ImportError:
    install('lightgbm')

try:
    import xgboost
except ImportError:
    install('xgboost')

# Core Libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# Machine Learning
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import (RandomForestRegressor, 
                               GradientBoostingRegressor,
                               HistGradientBoostingRegressor)
import lightgbm as lgb
import xgboost as xgb

# Settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.6f}'.format)
plt.style.use('seaborn-v0_8-darkgrid')

# Seed for reproducibility
SEED = 42
np.random.seed(SEED)

print('✅ All libraries loaded successfully!')
print(f'  LightGBM: {lightgbm.__version__}')
print(f'  XGBoost:  {xgb.__version__}')

In [ ]:
# ============================================================
# CELL 2: Load the Dataset
# ============================================================

# IMPORTANT: Update this path to wherever your dataset folder is
DATA_PATH = './dataset/'

train = pd.read_csv(DATA_PATH + 'train.csv')
test  = pd.read_csv(DATA_PATH + 'test.csv')
sample_submission = pd.read_csv(DATA_PATH + 'sample_submission.csv')

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')
print(f'Sample submission shape: {sample_submission.shape}')
print()
print('Train columns:', train.columns.tolist())
print('Test columns:', test.columns.tolist())

In [ ]:
# ============================================================
# CELL 3: Exploratory Data Analysis (EDA)
# ============================================================

print('=' * 60)
print('TRAIN DATA — First 5 rows')
print('=' * 60)
train.head()

In [ ]:
print('Missing values in TRAIN:')
missing_train = train.isnull().sum()
missing_train[missing_train > 0]

In [ ]:
print('Missing values in TEST:')
missing_test = test.isnull().sum()
missing_test[missing_test > 0]

In [ ]:
# Statistical summary of target variable
print('TARGET (demand) distribution:')
print(train['demand'].describe())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Target Variable (demand) Analysis', fontsize=16, fontweight='bold')

# Histogram
axes[0].hist(train['demand'], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Demand Distribution')
axes[0].set_xlabel('Demand')
axes[0].set_ylabel('Count')

# Box plot
axes[1].boxplot(train['demand'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[1].set_title('Demand Box Plot')
axes[1].set_ylabel('Demand')

# Log-scale histogram (demand is highly skewed)
axes[2].hist(np.log1p(train['demand']), bins=50, color='darkorange', edgecolor='white', alpha=0.8)
axes[2].set_title('log(1+Demand) Distribution')
axes[2].set_xlabel('log(1 + Demand)')

plt.tight_layout()
plt.show()

In [ ]:
# Impact of categorical features on demand
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Mean Demand by Categorical Features', fontsize=16, fontweight='bold')

cat_features = ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather']
colors_list = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

for ax, col, color in zip(axes.ravel(), cat_features, colors_list):
    data = train.groupby(col)['demand'].mean().sort_values(ascending=False)
    bars = ax.bar(data.index, data.values, color=color, alpha=0.8, edgecolor='white')
    ax.set_title(f'Mean Demand by {col}', fontsize=13)
    ax.set_ylabel('Mean Demand')
    # Add value labels on bars
    for bar, val in zip(bars, data.values):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Time-based demand patterns
train['hour_temp'] = train['timestamp'].apply(lambda x: int(x.split(':')[0]))
hourly_demand = train.groupby('hour_temp')['demand'].mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(hourly_demand.index, hourly_demand.values, 'o-', color='#4C72B0', 
        linewidth=2.5, markersize=8, markerfacecolor='white', markeredgewidth=2)
ax.fill_between(hourly_demand.index, hourly_demand.values, alpha=0.15, color='#4C72B0')
ax.set_title('Mean Traffic Demand by Hour of Day', fontsize=15, fontweight='bold')
ax.set_xlabel('Hour of Day', fontsize=12)
ax.set_ylabel('Mean Demand', fontsize=12)
ax.set_xticks(range(24))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Peak demand hours:')
print(hourly_demand.sort_values(ascending=False).head(5))
train.drop(columns=['hour_temp'], inplace=True)

In [ ]:
# Geohash analysis
print(f'Unique geohashes in train: {train["geohash"].nunique()}')
print(f'Unique geohashes in test:  {test["geohash"].nunique()}')
overlap = len(set(train['geohash']) & set(test['geohash']))
print(f'Geohash overlap: {overlap}')
print(f'Test geohashes NOT in train: {test["geohash"].nunique() - overlap}')
print()
print(f'Days in train: {sorted(train["day"].unique())}')
print(f'Days in test:  {sorted(test["day"].unique())}')
print()
day49_train_ts = train[train['day'] == 49]['timestamp'].unique()
day49_test_ts  = test[test['day'] == 49]['timestamp'].unique()
print(f'Day 49 train timestamps: {len(day49_train_ts)} (0:0 to 2:0)')
print(f'Day 49 test timestamps:  {len(day49_test_ts)} (2:15 to 13:45)')

In [ ]:
# ============================================================
# CELL 4: Feature Engineering
# ============================================================

print('Starting feature engineering...')

# ---- Step 1: Parse timestamp into numeric features ----
def parse_timestamp(df):
    """Convert 'H:MM' string into hour, minute, time_slot (0-95)."""
    df = df.copy()
    df['hour']      = df['timestamp'].apply(lambda x: int(x.split(':')[0]))
    df['minute']    = df['timestamp'].apply(lambda x: int(x.split(':')[1]))
    df['time_slot'] = df['hour'] * 4 + df['minute'] // 15  # 0 to 95 (96 slots per day)
    return df

train = parse_timestamp(train)
test  = parse_timestamp(test)

# ---- Step 2: Geohash prefix features (spatial hierarchy) ----
# Geohashes encode geographic locations: longer prefix = more specific area
for df in [train, test]:
    df['geo_prefix_4'] = df['geohash'].str[:4]  # Broader area
    df['geo_prefix_5'] = df['geohash'].str[:5]  # Medium area

# ---- Step 3: Compute historical aggregate statistics from TRAINING data ONLY ----
# IMPORTANT: All lookups must only use TRAINING data to prevent data leakage

# 3a. Most important: Historical demand at (geohash, time_slot)
#     This captures the "usual demand for this location at this time"
geo_ts_agg = (
    train.groupby(['geohash', 'time_slot'])['demand']
    .agg(['mean', 'std', 'median', 'min', 'max'])
    .reset_index()
    .rename(columns={
        'mean':   'geo_ts_mean',
        'std':    'geo_ts_std',
        'median': 'geo_ts_median',
        'min':    'geo_ts_min',
        'max':    'geo_ts_max'
    })
)

# 3b. Overall geohash statistics
geo_agg = (
    train.groupby('geohash')['demand']
    .agg(['mean', 'std', 'median', 'min', 'max', 'count'])
    .reset_index()
    .rename(columns={
        'mean':   'geo_mean',
        'std':    'geo_std',
        'median': 'geo_median',
        'min':    'geo_min',
        'max':    'geo_max',
        'count':  'geo_count'
    })
)

# 3c. Time slot statistics (how does demand change throughout the day?)
ts_agg = (
    train.groupby('time_slot')['demand']
    .agg(['mean', 'std'])
    .reset_index()
    .rename(columns={'mean': 'ts_mean', 'std': 'ts_std'})
)

# 3d. Hour-level statistics
hour_agg = (
    train.groupby('hour')['demand']
    .mean()
    .reset_index()
    .rename(columns={'demand': 'hour_mean'})
)

# 3e. Geohash-prefix level statistics (for geohashes not in training)
geo4_agg = (
    train.groupby('geo_prefix_4')['demand']
    .agg(['mean', 'std'])
    .reset_index()
    .rename(columns={'mean': 'geo4_mean', 'std': 'geo4_std'})
)

geo5_agg = (
    train.groupby('geo_prefix_5')['demand']
    .agg(['mean', 'std'])
    .reset_index()
    .rename(columns={'mean': 'geo5_mean', 'std': 'geo5_std'})
)

# 3f. RoadType aggregates
road_agg = (
    train.groupby('RoadType')['demand']
    .mean()
    .reset_index()
    .rename(columns={'demand': 'roadtype_mean'})
)

# 3g. (geohash, hour) combination — captures location-specific daily cycle
geo_hour_agg = (
    train.groupby(['geohash', 'hour'])['demand']
    .mean()
    .reset_index()
    .rename(columns={'demand': 'geo_hour_mean'})
)

# 3h. (geo_prefix_5, time_slot) — for fallback when exact geohash is unknown
geo5_ts_agg = (
    train.groupby(['geo_prefix_5', 'time_slot'])['demand']
    .mean()
    .reset_index()
    .rename(columns={'demand': 'geo5_ts_mean'})
)

# 3i. (RoadType, time_slot) interaction
road_ts_agg = (
    train.groupby(['RoadType', 'time_slot'])['demand']
    .mean()
    .reset_index()
    .rename(columns={'demand': 'road_ts_mean'})
)

# 3j. (RoadType, hour) interaction 
road_hour_agg = (
    train.groupby(['RoadType', 'hour'])['demand']
    .mean()
    .reset_index()
    .rename(columns={'demand': 'road_hour_mean'})
)

# 3k. Number of lanes + RoadType interaction
lanes_road_agg = (
    train.groupby(['NumberofLanes', 'RoadType'])['demand']
    .mean()
    .reset_index()
    .rename(columns={'demand': 'lanes_road_mean'})
)

# 3l. Day-of-week demand scale (day 48 vs day 49)
day_agg = (
    train.groupby('day')['demand']
    .mean()
    .reset_index()
    .rename(columns={'demand': 'day_mean'})
)

print('✅ Aggregate features computed!')
print(f'  geo_ts_agg shape:      {geo_ts_agg.shape}')
print(f'  geo_agg shape:         {geo_agg.shape}')
print(f'  geo5_ts_agg shape:     {geo5_ts_agg.shape}')

In [ ]:
# ============================================================
# CELL 5: Merge features and encode categoricals
# ============================================================

# Global fallback (if even prefix lookups fail)
GLOBAL_MEAN = train['demand'].mean()

def merge_features(df):
    """Merge all pre-computed aggregate features into the dataframe."""
    df = df.copy()
    
    # Merge all lookup tables
    df = df.merge(geo_ts_agg,      on=['geohash', 'time_slot'], how='left')
    df = df.merge(geo_agg,         on='geohash',                how='left')
    df = df.merge(ts_agg,          on='time_slot',              how='left')
    df = df.merge(hour_agg,        on='hour',                   how='left')
    df = df.merge(geo4_agg,        on='geo_prefix_4',           how='left')
    df = df.merge(geo5_agg,        on='geo_prefix_5',           how='left')
    df = df.merge(road_agg,        on='RoadType',               how='left')
    df = df.merge(geo_hour_agg,    on=['geohash', 'hour'],      how='left')
    df = df.merge(geo5_ts_agg,     on=['geo_prefix_5','time_slot'], how='left')
    df = df.merge(road_ts_agg,     on=['RoadType','time_slot'], how='left')
    df = df.merge(road_hour_agg,   on=['RoadType','hour'],      how='left')
    df = df.merge(lanes_road_agg,  on=['NumberofLanes','RoadType'], how='left')
    df = df.merge(day_agg,         on='day',                    how='left')
    
    return df

train_feat = merge_features(train)
test_feat  = merge_features(test)

print('Features merged. Now encoding categoricals...')

# ---- Encode categorical columns ----
# RoadType: Residential=0, Street=1, Highway=2 (ordered by mean demand)
ROADTYPE_MAP = {'Residential': 0, 'Street': 1, 'Highway': 2}
# LargeVehicles: binary
LARGEVEHICLES_MAP = {'Not Allowed': 0, 'Allowed': 1}
# Landmarks: binary
LANDMARKS_MAP = {'No': 0, 'Yes': 1}
# Weather: nominal
WEATHER_MAP = {'Sunny': 0, 'Rainy': 1, 'Foggy': 2, 'Snowy': 3}

for df in [train_feat, test_feat]:
    df['RoadType_enc']      = df['RoadType'].map(ROADTYPE_MAP).fillna(-1)
    df['LargeVehicles_enc'] = df['LargeVehicles'].map(LARGEVEHICLES_MAP).fillna(-1)
    df['Landmarks_enc']     = df['Landmarks'].map(LANDMARKS_MAP).fillna(-1)
    df['Weather_enc']       = df['Weather'].map(WEATHER_MAP).fillna(-1)

# ---- Fill Temperature missing values ----
# Strategy: fill with global median (temperature has near-zero correlation with demand)
TEMP_MEDIAN = train['Temperature'].median()
for df in [train_feat, test_feat]:
    df['Temperature'] = df['Temperature'].fillna(TEMP_MEDIAN)

# ---- Cyclical time encoding (sin/cos for hour and time_slot) ----
# This is crucial: hour 23 and hour 0 are close in time, but far numerically
for df in [train_feat, test_feat]:
    df['hour_sin']  = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos']  = np.cos(2 * np.pi * df['hour'] / 24)
    df['ts_sin']    = np.sin(2 * np.pi * df['time_slot'] / 96)
    df['ts_cos']    = np.cos(2 * np.pi * df['time_slot'] / 96)
    df['minute_sin'] = np.sin(2 * np.pi * df['minute'] / 60)
    df['minute_cos'] = np.cos(2 * np.pi * df['minute'] / 60)

print('✅ Feature engineering complete!')

In [ ]:
# ============================================================
# CELL 6: Define Feature Set and Prepare Data
# ============================================================

# Feature list — carefully selected based on EDA and importance analysis
FEATURES = [
    # Geo-temporal historical demand (MOST IMPORTANT)
    'geo_ts_mean',       # Avg demand at this exact (location, time)
    'geo_ts_std',        # Variability of demand at this exact (location, time)
    'geo_ts_median',     # Median demand at this exact (location, time)
    'geo_ts_min',        # Min demand at this exact (location, time)
    'geo_ts_max',        # Max demand at this exact (location, time)
    
    # Geohash aggregate features
    'geo_mean',          # Overall mean demand for this location
    'geo_std',           # Demand variability for this location
    'geo_median',        # Median demand for this location
    'geo_min',
    'geo_max',
    'geo_count',         # How many observations we have for this location
    
    # Spatial fallbacks (when exact geohash is unseen)
    'geo4_mean',         # Mean demand for 4-char prefix (broader area)
    'geo4_std',
    'geo5_mean',         # Mean demand for 5-char prefix (narrower area)
    'geo5_std',
    
    # Temporal features
    'ts_mean',           # Average demand for this time slot across all locations
    'ts_std',
    'hour_mean',         # Average demand for this hour
    
    # Interaction features
    'geo_hour_mean',     # Location × hour interaction
    'geo5_ts_mean',      # Area-prefix × time interaction
    'road_ts_mean',      # Road type × time interaction
    'road_hour_mean',    # Road type × hour interaction
    'lanes_road_mean',   # Lanes × road type interaction
    'roadtype_mean',     # Mean demand for this road type
    'day_mean',          # Day-level average
    
    # Raw time features
    'time_slot',         # Discrete time slot (0-95)
    'hour',              # Hour of day
    'minute',            # Minute within hour
    
    # Cyclical encoding
    'hour_sin', 'hour_cos',
    'ts_sin', 'ts_cos',
    'minute_sin', 'minute_cos',
    
    # Encoded categorical features
    'RoadType_enc',
    'LargeVehicles_enc',
    'Landmarks_enc',
    'Weather_enc',
    
    # Numerical features
    'Temperature',
    'NumberofLanes',
    'day',
]

X_train = train_feat[FEATURES].copy()
y_train = train_feat['demand'].copy()
X_test  = test_feat[FEATURES].copy()

# Fill any remaining NaN in features with -999 (tree models handle this)
X_train = X_train.fillna(-999)
X_test  = X_test.fillna(-999)

print(f'Training features shape: {X_train.shape}')
print(f'Test features shape:     {X_test.shape}')
print(f'Target shape:            {y_train.shape}')
print()
print('NaN check — X_train:', X_train.isnull().sum().sum())
print('NaN check — X_test: ', X_test.isnull().sum().sum())

In [ ]:
# ============================================================
# CELL 7: Model 1 — LightGBM (primary model)
# ============================================================

print('Training LightGBM model...')
print('=' * 60)

# LightGBM hyperparameters — tuned for regression with small dataset
lgb_params = {
    'objective':        'regression',        # We're doing regression (predicting demand float)
    'metric':           'rmse',             # Minimize RMSE during training
    'boosting_type':    'gbdt',             # Gradient Boosting Decision Tree
    'num_leaves':       127,                # Complexity: max leaves per tree
    'max_depth':        -1,                 # No limit on tree depth
    'learning_rate':    0.05,               # Step size — smaller = slower but more accurate
    'n_estimators':     1000,               # Number of trees to build
    'subsample':        0.8,                # Use 80% of rows per tree (prevents overfitting)
    'colsample_bytree': 0.8,                # Use 80% of features per tree
    'min_child_samples': 20,               # Minimum samples in a leaf
    'reg_alpha':        0.01,              # L1 regularization
    'reg_lambda':       0.01,              # L2 regularization
    'random_state':     SEED,
    'n_jobs':           -1,                # Use all CPU cores
    'verbose':          -1,                # Suppress output
}

# K-Fold Cross Validation — split training data into 5 folds
# Each fold: train on 4 parts, validate on 1 part → gives reliable performance estimate
N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

lgb_oof_preds   = np.zeros(len(X_train))   # Out-of-Fold predictions (for stacking later)
lgb_test_preds  = np.zeros(len(X_test))    # Final test predictions (average of all folds)
lgb_r2_scores   = []                       # Store R² for each fold
lgb_importances = pd.DataFrame()

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train, y_train), 1):
    print(f'  Fold {fold}/{N_FOLDS}...', end=' ')
    
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    model = lgb.LGBMRegressor(**lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(period=-1)]
    )
    
    val_pred = model.predict(X_val, num_iteration=model.best_iteration_)
    val_pred = np.clip(val_pred, 0, 1)   # Clip to valid demand range
    
    fold_r2 = r2_score(y_val, val_pred)
    lgb_r2_scores.append(fold_r2)
    lgb_oof_preds[val_idx] = val_pred
    lgb_test_preds += model.predict(X_test, num_iteration=model.best_iteration_) / N_FOLDS
    
    # Collect feature importances
    fold_importance = pd.DataFrame({
        'feature':    FEATURES,
        'importance': model.feature_importances_,
        'fold':       fold
    })
    lgb_importances = pd.concat([lgb_importances, fold_importance], ignore_index=True)
    
    print(f'R² = {fold_r2:.4f}, Score = {max(0, 100*fold_r2):.2f}')

lgb_test_preds = np.clip(lgb_test_preds, 0, 1)
lgb_oof_r2     = r2_score(y_train, lgb_oof_preds)

print()
print(f'LightGBM — Individual fold R² scores: {[round(s, 4) for s in lgb_r2_scores]}')
print(f'LightGBM — Mean fold R²: {np.mean(lgb_r2_scores):.4f} ± {np.std(lgb_r2_scores):.4f}')
print(f'LightGBM — OOF R²:       {lgb_oof_r2:.4f}')
print(f'LightGBM — OOF Score:    {max(0, 100*lgb_oof_r2):.2f}')

In [ ]:
# ============================================================
# CELL 8: LightGBM Feature Importance
# ============================================================

# Average feature importance across all folds
mean_importance = (
    lgb_importances.groupby('feature')['importance']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(14, 10))
top_n = 25
top_features = mean_importance.head(top_n)

colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, top_n))[::-1]
bars = ax.barh(range(top_n), top_features['importance'], color=colors, edgecolor='white')

ax.set_yticks(range(top_n))
ax.set_yticklabels(top_features['feature'], fontsize=11)
ax.invert_yaxis()
ax.set_xlabel('Mean Importance (across 5 folds)', fontsize=12)
ax.set_title(f'Top {top_n} Feature Importances — LightGBM', fontsize=15, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# Add value labels
for bar, val in zip(bars, top_features['importance']):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2.,
            f'{val:.0f}', ha='left', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print('\nTop 10 features:')
print(mean_importance.head(10).to_string(index=False))

In [ ]:
# ============================================================
# CELL 9: Model 2 — XGBoost (secondary model for ensemble)
# ============================================================

print('Training XGBoost model...')
print('=' * 60)

xgb_params = {
    'objective':          'reg:squarederror',  # Regression with MSE loss
    'eval_metric':        'rmse',
    'max_depth':          6,                   # Tree depth
    'learning_rate':      0.05,
    'n_estimators':       1000,
    'subsample':          0.8,
    'colsample_bytree':   0.8,
    'min_child_weight':   5,
    'gamma':              0.01,                # Minimum loss reduction for split
    'reg_alpha':          0.01,
    'reg_lambda':         1.0,
    'random_state':       SEED,
    'n_jobs':             -1,
    'verbosity':          0,
}

xgb_oof_preds  = np.zeros(len(X_train))
xgb_test_preds = np.zeros(len(X_test))
xgb_r2_scores  = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train, y_train), 1):
    print(f'  Fold {fold}/{N_FOLDS}...', end=' ')
    
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    model = xgb.XGBRegressor(**xgb_params, early_stopping_rounds=50)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    
    val_pred = model.predict(X_val)
    val_pred = np.clip(val_pred, 0, 1)
    
    fold_r2 = r2_score(y_val, val_pred)
    xgb_r2_scores.append(fold_r2)
    xgb_oof_preds[val_idx] = val_pred
    xgb_test_preds += model.predict(X_test) / N_FOLDS
    
    print(f'R² = {fold_r2:.4f}, Score = {max(0, 100*fold_r2):.2f}')

xgb_test_preds = np.clip(xgb_test_preds, 0, 1)
xgb_oof_r2     = r2_score(y_train, xgb_oof_preds)

print()
print(f'XGBoost — Mean fold R²: {np.mean(xgb_r2_scores):.4f} ± {np.std(xgb_r2_scores):.4f}')
print(f'XGBoost — OOF R²:       {xgb_oof_r2:.4f}')
print(f'XGBoost — OOF Score:    {max(0, 100*xgb_oof_r2):.2f}')

In [ ]:
# ============================================================
# CELL 10: Model 3 — HistGradientBoostingRegressor (tertiary)
# ============================================================

print('Training Hist Gradient Boosting model...')
print('=' * 60)

hgb_params = {
    'max_iter':          1000,
    'max_leaf_nodes':    63,
    'max_depth':         None,
    'learning_rate':     0.05,
    'l2_regularization': 0.01,
    'min_samples_leaf':  20,
    'random_state':      SEED,
    'early_stopping':    True,
    'n_iter_no_change':  50,
    'validation_fraction': 0.1,
}

hgb_oof_preds  = np.zeros(len(X_train))
hgb_test_preds = np.zeros(len(X_test))
hgb_r2_scores  = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train, y_train), 1):
    print(f'  Fold {fold}/{N_FOLDS}...', end=' ')
    
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    # Replace -999 with NaN — HistGBR handles NaN natively
    X_tr_hgb  = X_tr.replace(-999, np.nan)
    X_val_hgb = X_val.replace(-999, np.nan)
    X_test_hgb = X_test.replace(-999, np.nan)
    
    model = HistGradientBoostingRegressor(**hgb_params)
    model.fit(X_tr_hgb, y_tr)
    
    val_pred = model.predict(X_val_hgb)
    val_pred = np.clip(val_pred, 0, 1)
    
    fold_r2 = r2_score(y_val, val_pred)
    hgb_r2_scores.append(fold_r2)
    hgb_oof_preds[val_idx] = val_pred
    hgb_test_preds += model.predict(X_test_hgb) / N_FOLDS
    
    print(f'R² = {fold_r2:.4f}, Score = {max(0, 100*fold_r2):.2f}')

hgb_test_preds = np.clip(hgb_test_preds, 0, 1)
hgb_oof_r2     = r2_score(y_train, hgb_oof_preds)

print()
print(f'HistGBR — Mean fold R²: {np.mean(hgb_r2_scores):.4f} ± {np.std(hgb_r2_scores):.4f}')
print(f'HistGBR — OOF R²:       {hgb_oof_r2:.4f}')
print(f'HistGBR — OOF Score:    {max(0, 100*hgb_oof_r2):.2f}')

In [ ]:
# ============================================================
# CELL 11: Ensemble — Weighted Average of All Models
# ============================================================

print('Creating ensemble predictions...')
print('=' * 60)

# Find optimal ensemble weights using OOF predictions
# Strategy: weight each model proportionally to its OOF R² score
from scipy.optimize import minimize

def ensemble_objective(weights):
    """Minimize negative R² for ensemble (i.e., maximize R²)."""
    w = np.array(weights)
    w = w / w.sum()  # Normalize weights to sum to 1
    preds = (w[0] * lgb_oof_preds + 
             w[1] * xgb_oof_preds + 
             w[2] * hgb_oof_preds)
    return -r2_score(y_train, np.clip(preds, 0, 1))

# Start with equal weights and optimize
initial_weights = [1.0, 1.0, 1.0]
result = minimize(
    ensemble_objective,
    initial_weights,
    method='Nelder-Mead',
    options={'maxiter': 500, 'xatol': 1e-6, 'fatol': 1e-6}
)

optimal_weights = np.array(result.x)
optimal_weights = optimal_weights / optimal_weights.sum()  # Normalize

print(f'Optimal ensemble weights:')
print(f'  LightGBM:  {optimal_weights[0]:.4f}')
print(f'  XGBoost:   {optimal_weights[1]:.4f}')
print(f'  HistGBR:   {optimal_weights[2]:.4f}')
print()

# Compute ensemble OOF predictions
ensemble_oof = np.clip(
    optimal_weights[0] * lgb_oof_preds +
    optimal_weights[1] * xgb_oof_preds +
    optimal_weights[2] * hgb_oof_preds,
    0, 1
)

ensemble_r2 = r2_score(y_train, ensemble_oof)

print(f'Ensemble OOF R²:    {ensemble_r2:.4f}')
print(f'Ensemble OOF Score: {max(0, 100*ensemble_r2):.2f}')
print()
print('Individual model scores:')
print(f'  LightGBM OOF Score: {max(0, 100*lgb_oof_r2):.2f}')
print(f'  XGBoost OOF Score:  {max(0, 100*xgb_oof_r2):.2f}')
print(f'  HistGBR OOF Score:  {max(0, 100*hgb_oof_r2):.2f}')

# Final test predictions
final_preds = np.clip(
    optimal_weights[0] * lgb_test_preds +
    optimal_weights[1] * xgb_test_preds +
    optimal_weights[2] * hgb_test_preds,
    0, 1
)

print(f'\nFinal prediction stats:')
print(f'  Min:  {final_preds.min():.6f}')
print(f'  Max:  {final_preds.max():.6f}')
print(f'  Mean: {final_preds.mean():.6f}')

In [ ]:
# ============================================================
# CELL 12: Residual Analysis & Visualization
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('Model Performance Analysis', fontsize=18, fontweight='bold')

# 1. Actual vs Predicted (scatter)
sample_idx = np.random.choice(len(y_train), size=5000, replace=False)
ax = axes[0, 0]
ax.scatter(y_train.iloc[sample_idx], ensemble_oof[sample_idx], 
           alpha=0.3, s=5, color='#4C72B0')
ax.plot([0, 1], [0, 1], 'r--', lw=2, label='Perfect prediction')
ax.set_xlabel('Actual Demand', fontsize=12)
ax.set_ylabel('Predicted Demand', fontsize=12)
ax.set_title(f'Actual vs Predicted (R²={ensemble_r2:.4f})', fontsize=13)
ax.legend()

# 2. Residuals distribution
residuals = y_train.values - ensemble_oof
ax = axes[0, 1]
ax.hist(residuals, bins=80, color='#DD8452', alpha=0.8, edgecolor='white')
ax.axvline(0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Residual (Actual - Predicted)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Residuals Distribution', fontsize=13)
ax.set_xlim(-0.5, 0.5)

# 3. Model comparison
ax = axes[0, 2]
models = ['LightGBM', 'XGBoost', 'HistGBR', 'Ensemble']
scores = [
    max(0, 100*lgb_oof_r2),
    max(0, 100*xgb_oof_r2),
    max(0, 100*hgb_oof_r2),
    max(0, 100*ensemble_r2)
]
colors_bar = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
bars = ax.bar(models, scores, color=colors_bar, alpha=0.85, edgecolor='white')
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
            f'{score:.2f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
ax.set_ylabel('Score (max(0, 100*R²))', fontsize=12)
ax.set_title('Model Score Comparison', fontsize=13)
ax.set_ylim(0, max(scores) * 1.15)

# 4. Predicted demand distribution (test)
ax = axes[1, 0]
ax.hist(final_preds, bins=50, color='#55A868', alpha=0.8, edgecolor='white', label='Test predictions')
ax.hist(y_train, bins=50, color='#4C72B0', alpha=0.4, edgecolor='white', label='Train actual')
ax.set_xlabel('Demand', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Train vs Test Demand Distributions', fontsize=13)
ax.legend()

# 5. Residuals vs Predicted
ax = axes[1, 1]
ax.scatter(ensemble_oof[sample_idx], residuals[sample_idx], 
           alpha=0.3, s=5, color='#9467BD')
ax.axhline(0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Predicted Demand', fontsize=12)
ax.set_ylabel('Residual', fontsize=12)
ax.set_title('Residuals vs Predicted', fontsize=13)
ax.set_ylim(-0.5, 0.5)

# 6. CV Fold scores
ax = axes[1, 2]
fold_nums = list(range(1, N_FOLDS + 1))
ax.plot(fold_nums, [max(0, 100*s) for s in lgb_r2_scores], 'o-', label='LightGBM', color='#4C72B0', lw=2)
ax.plot(fold_nums, [max(0, 100*s) for s in xgb_r2_scores], 's-', label='XGBoost', color='#DD8452', lw=2)
ax.plot(fold_nums, [max(0, 100*s) for s in hgb_r2_scores], '^-', label='HistGBR', color='#55A868', lw=2)
ax.set_xlabel('Fold', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('CV Fold Scores', fontsize=13)
ax.legend()
ax.set_xticks(fold_nums)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 13: Generate Submission File
# ============================================================

# Create the submission dataframe
submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': final_preds
})

# Verify submission format matches sample_submission
print('Submission shape:', submission.shape)
print('Expected shape:  (41778, 2)')
print()
print('Submission columns:', submission.columns.tolist())
print('Expected columns:  ["Index", "demand"]')
print()
print('First 5 rows:')
print(submission.head())
print()
print('Submission demand stats:')
print(submission['demand'].describe())
print()

# Check no out-of-range values
print(f'Values below 0: {(submission["demand"] < 0).sum()}')
print(f'Values above 1: {(submission["demand"] > 1).sum()}')
print(f'NaN values:     {submission["demand"].isna().sum()}')

# Save submission
SUBMISSION_PATH = './submission.csv'
submission.to_csv(SUBMISSION_PATH, index=False)
print(f'\n✅ Submission saved to: {SUBMISSION_PATH}')

In [ ]:
# ============================================================
# CELL 14: Verify Submission Against sample_submission.csv
# ============================================================

sample_sub = pd.read_csv(DATA_PATH + 'sample_submission.csv')
our_sub    = pd.read_csv(SUBMISSION_PATH)

print('Submission column check:')
print(f'  Required columns: {sample_sub.columns.tolist()}')
print(f'  Our columns:      {our_sub.columns.tolist()}')
print(f'  Match: {list(sample_sub.columns) == list(our_sub.columns)}')
print()
print('Index check:')
# Check that all required indices from test are present
test_indices = set(test['Index'].values)
our_indices  = set(our_sub['Index'].values)
print(f'  Test indices:      {len(test_indices)}')
print(f'  Submission indices:{len(our_indices)}')
print(f'  All match: {test_indices == our_indices}')
print()
print('✅ Submission file is valid!')
print('\nFinal submission preview:')
our_sub.head(10)

In [ ]:
# ============================================================
# CELL 15: Summary and Final Score Estimation
# ============================================================

print('=' * 60)
print('FINAL RESULTS SUMMARY')
print('=' * 60)
print()
print(f'📊 Model Performance (OOF = Out-of-Fold Cross Validation):')
print(f'   LightGBM OOF Score:  {max(0, 100*lgb_oof_r2):>7.2f}')
print(f'   XGBoost OOF Score:   {max(0, 100*xgb_oof_r2):>7.2f}')
print(f'   HistGBR OOF Score:   {max(0, 100*hgb_oof_r2):>7.2f}')
print(f'   ✨ Ensemble OOF Score:{max(0, 100*ensemble_r2):>7.2f}')
print()
print(f'📁 Submission file: ./submission.csv ({len(submission)} rows)')
print(f'   Index column:  ✅ Present')
print(f'   demand column: ✅ Present')
print(f'   Shape: {submission.shape} (expected: 41778 × 2)')
print()
print('🔑 Key insights from the data:')
print('   1. geo_ts_mean (historical demand at same location+time) is the #1 feature')
print('   2. RoadType is the most impactful categorical (Highway >> Street >> Residential)')
print('   3. Demand follows a daily cycle peaking in mid-morning hours')
print('   4. Ensemble of 3 gradient boosting models outperforms any single model')